**Clonning the repo to use the dataset**

In [1]:
!git clone https://github.com/amanmandal7/COMP8410-A1_Evaluation_Presentation.git
%cd COMP8410-A1_Evaluation_Presentation/pair4_multimodal

fatal: destination path 'COMP8410-A1_Evaluation_Presentation' already exists and is not an empty directory.
/content/COMP8410-A1_Evaluation_Presentation/pair4_multimodal


**Installing and Laoding CLIP**

In [2]:
!pip install transformers -q
from transformers import pipeline
import os

clip = pipeline("zero-shot-image-classification", model="openai/clip-vit-base-patch32")
print("CLIP loaded")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIP loaded


**Defining ground-truth labels**

In [3]:
# the correct label for each image (what we know it actually is)
ground_truth = {
    "passport_1.png": "a passport",
    "passport_2.jpg": "a passport",
    "license_1.jpg": "a driver's licence",
    "license_2.png": "a driver's licence",
    "idcard_1.png": "an ID card",
    "selfie_1.jpeg": "a photo of a person",
    "selfie_2.jpeg": "a photo of a person",
    "random_1.webp": "a random object",
    "random_2.jpg": "a random object",
    "random_3.jfif": "a random object",
}

# the candidate labels CLIP chooses from
candidate_labels = ["a passport", "a driver's licence", "an ID card",
                    "a photo of a person", "a random object"]

# Model A: Document-type classifier

In [4]:
data_dir = "data"
correct_preds = 0

for fname, true_label in ground_truth.items():
    path = os.path.join(data_dir, fname)

    # run clip zero-shot classification
    result = clip(path, candidate_labels=candidate_labels)
    predicted = result[0]["label"]

    is_match = (predicted == true_label)
    if is_match:
        correct_preds += 1

    status = "correct" if is_match else "incorrect"
    print(f"{fname}: predicted '{predicted}' | true '{true_label}' | {status}")

accuracy_a = correct_preds / len(ground_truth)
print(f"\nModel A accuracy: {accuracy_a:.2f} ({correct_preds}/{len(ground_truth)})")

passport_1.png: predicted 'a passport' | true 'a passport' | correct
passport_2.jpg: predicted 'a driver's licence' | true 'a passport' | incorrect
license_1.jpg: predicted 'a driver's licence' | true 'a driver's licence' | correct
license_2.png: predicted 'a driver's licence' | true 'a driver's licence' | correct
idcard_1.png: predicted 'a driver's licence' | true 'an ID card' | incorrect
selfie_1.jpeg: predicted 'a photo of a person' | true 'a photo of a person' | correct
selfie_2.jpeg: predicted 'a photo of a person' | true 'a photo of a person' | correct
random_1.webp: predicted 'a driver's licence' | true 'a random object' | incorrect
random_2.jpg: predicted 'a random object' | true 'a random object' | correct
random_3.jfif: predicted 'a photo of a person' | true 'a random object' | incorrect

Model A accuracy: 0.60 (6/10)


# Model B: Claim-consistency check

In [5]:
tests = [
    ("passport_1.png", "a passport", "a driver's licence"),
    ("selfie_1.jpeg", "a photo of a person", "a passport"),
    ("license_1.jpg", "a driver's licence", "a passport"),
]

for fname, true_claim, false_claim in tests:
    img_path = os.path.join(data_dir, fname)

    if not os.path.exists(img_path):
        print(f"Warning: {fname} not found in {data_dir}, skipping...")
        continue

    res = clip(img_path, candidate_labels=[true_claim, false_claim])

    scores = {}
    for r in res:
        scores[r["label"]] = r["score"]

    # printing out comparison
    t_score = scores.get(true_claim, 0.0)
    f_score = scores.get(false_claim, 0.0)

    print(f"{fname}: true claim '{true_claim}' score={t_score:.2f} | false claim '{false_claim}' score={f_score:.2f}")

passport_1.png: true claim 'a passport' score=0.71 | false claim 'a driver's licence' score=0.29
selfie_1.jpeg: true claim 'a photo of a person' score=0.93 | false claim 'a passport' score=0.07
license_1.jpg: true claim 'a driver's licence' score=0.97 | false claim 'a passport' score=0.03
